# 05 · Tam eğitim (Adım 8) — ChangeOS, ön-kayıtlı
**Ön-kayıt:** `docs/adim8_tam_egitim_onkayit.md` (29 Ekim 2026, sonuçlardan ÖNCE). Bu not defteri **hiçbir koşuda test bölmesine dokunmaz**; test, tüm eğitimler bittikten sonra ayrı bir not defterinde tek turda değerlendirilir.

**Kaggle ayarları (sağ panel):** Accelerator **GPU T4 x2** · Internet **On** · Secrets: `GITHUB_TOKEN` **Attached** · Input: dataset `melihkotman/disasterm3-turkiye` (isteğe bağlı yedek: `kod` dataset'i).

**Kullanım**
1. `MODE = "dry"` → *Save Version → Save & Run All (Commit)*. ≈15–25 dk GPU. Sonunda "KURU KOŞU ÖZETİ" hücresini oku (paralel iki GPU verimli mi? kilitler çalışıyor mu?). Kota göstergesine de bak (aşağıda).
2. Sorun yoksa `MODE = "run"` yap, yine *Save & Run All (Commit)*. Koşu Kaggle sunucusunda sürer; bilgisayar kapansa/internet gitse devam eder. Çıktı sürümün **Output** sekmesinde kalır.
3. Paralel verimsizse `PARALLEL = False` ve şeritleri ayrı sürümlerde koş (`RUN_LANES = ["A"]`, sonra `["B"]`).


In [ ]:
# ---- PARAMETRELER (tek yerden) ----
MODE = "dry"                 # "dry" = kuru koşu | "run" = tam eğitim
CODE_REF = "adim8-full-v1"   # push edilmiş git etiketi; her koşu bu sürümle yapılır
RUN_LANES = ["A", "B"]       # run modunda hangi şeritler
PARALLEL = True              # True: şeritler iki GPU'da aynı anda. Kuru koşu verimli göstermeden True bırakma.

# (ad, senaryo, seed, tetikleyici_katlaması)  — ön-kayıt §3; ayar YOK, 50 epoch, son epoch ağırlığı
LANES = {
    "A": [("s2_seed42", "S2", 42, False), ("s1_seed42", "S1", 42, False)],                                   # ≈ 5,3 sa
    "B": [("trigger_osmaniye_seed42", "S1", 42, True), ("s1_seed43", "S1", 43, False), ("s1_seed44", "S1", 44, False)],  # ≈ 6,1 sa
}
EPOCHS = 50
print("MODE:", MODE, "| CODE_REF:", CODE_REF, "| RUN_LANES:", RUN_LANES, "| PARALLEL:", PARALLEL)


In [ ]:
# 1) Yardımcılar + kurulum (ever-beta 0.5.6 sabit!)
import os, sys, json, time, glob, hashlib, subprocess, threading, shutil

def sh(cmd, check=True, mask=(), tail=3000):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    for m in mask:
        out = out.replace(m, "***")
    if out.strip():
        print(out.strip()[-tail:])
    if check and r.returncode:
        raise SystemExit(f"komut hata verdi (çıkış {r.returncode}); çıktıya bak")
    return r.returncode

sh('pip install -q timm "ever-beta==0.5.6" albumentations tifffile einops "datasets[vision]"')
sh("pip install -q --no-deps git+https://github.com/Z-Zheng/pytorch-change-models.git")
sh("nvidia-smi --query-gpu=index,name,memory.total --format=csv")


In [ ]:
# 2) Kodu al, etikete sabitle, sürümü kaydet
REPO = "/kaggle/working/deprem-vlm-rag"

def get_code():
    if os.path.exists(REPO):
        return
    tok = None
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception as e:
        print("GITHUB_TOKEN okunamadı:", type(e).__name__)
    if tok:
        rc = sh(f"git clone -q https://{tok}@github.com/MelihKotman/Deprem-VLM-RAG.git {REPO}", check=False, mask=(tok,))
        if rc == 0:
            sh(f"git -C {REPO} remote set-url origin https://github.com/MelihKotman/Deprem-VLM-RAG.git")
    if not os.path.exists(REPO):
        cand = glob.glob("/kaggle/input/**/deprem-vlm-rag", recursive=True)
        if not cand:
            raise SystemExit("kod alınamadı: GITHUB_TOKEN 'Attached' mı? ya da 'kod' dataset'ini ekle")
        shutil.copytree(cand[0], REPO)
        print("UYARI: kod 'kod' dataset'inden alındı (git etiketi doğrulanamaz)")

get_code()
git_sha = None
if os.path.exists(f"{REPO}/.git"):
    sh(f"git -C {REPO} fetch -q --tags", check=False)
    if sh(f"git -C {REPO} checkout -q {CODE_REF}", check=False):
        raise SystemExit(f"CODE_REF '{CODE_REF}' bulunamadı: yerelde `git tag {CODE_REF}` + `git push origin {CODE_REF}` yap")
    git_sha = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
files = sorted(glob.glob(f"{REPO}/src/*.py")) + [f"{REPO}/config/base.yaml"]
src_sha1 = hashlib.sha1(b"".join(open(f, "rb").read() for f in files)).hexdigest()
pipv = subprocess.run("pip freeze 2>/dev/null | grep -i -E '^(torch|torchvision|ever-beta|torchange|timm|numpy|opencv)'", shell=True, capture_output=True, text=True).stdout
with open("/kaggle/working/code_version.txt", "w") as f:
    f.write(f"code_ref={CODE_REF}\ngit_sha={git_sha}\nsrc_config_sha1={src_sha1}\n{pipv}")
print(open("/kaggle/working/code_version.txt").read())
os.chdir(REPO); sys.path.insert(0, REPO)
sh("python -m src.config && python -m src.dm3_verify", tail=1500)
sh("python -m src.selftest_eval", tail=800)


In [ ]:
# 3) Koşu yardımcıları (alt süreç + günlük dosyası; ekrana yalnız ilerleme/hata satırları)
RUNS = "/kaggle/working/runs"; LOGS = "/kaggle/working/logs"
os.makedirs(RUNS, exist_ok=True); os.makedirs(LOGS, exist_ok=True)
CACHE_ROOT = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"   # çıktıya (Output) girmesin
print("önbellek kökü:", CACHE_ROOT); sh(f"df -h {CACHE_ROOT} | tail -1")
SHOW = ("İLERLEME", "Traceback", "Error", "error", "timing:", "tetikleyici:", "metrikler", "out of memory", "EXTRAPOLATION", "[S1]", "[S2]", "--max-minutes", "kilitli", "ayrılmıştır")

def run_job(tag, args, gpu, cache, workers, module="src.train_changeos"):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
    cmd = [sys.executable, "-m", module, *args, "--cache-dir", cache, "--workers", str(workers)]
    oom = False
    with open(f"{LOGS}/{tag}.log", "a") as log:
        p = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line)
            if "out of memory" in line.lower():
                oom = True
            if any(k in line for k in SHOW):
                print(f"[{tag}] {line.rstrip()[:260]}", flush=True)
        p.wait()
    return p.returncode, oom

def train_job(job, gpu, cache, workers):
    name, scen, seed, trig = job
    out = f"{RUNS}/{name}"
    if os.path.exists(f"{out}/model_final.pt") and os.path.exists(f"{out}/metrics.json"):
        print(f"[{name}] zaten tamam, atlanıyor"); return 0
    base = ["--scenario", scen, "--epochs", str(EPOCHS), "--seed", str(seed), "--max-minutes", "600", "--out", out]
    if trig:
        base += ["--holdout", "Osmaniye", "--trigger-fold"]
    extra = []
    for attempt in range(3):
        args = base + extra + (["--resume"] if os.path.exists(f"{out}/last.pt") else [])
        rc, oom = run_job(name, args, gpu, cache, workers)
        if rc == 0:
            return 0
        if rc == 3:                       # --max-minutes: kaldığı yerden devam
            continue
        if oom and "--no-cudnn-benchmark" not in extra:
            extra = ["--no-cudnn-benchmark"]; print(f"[{name}] OOM → --no-cudnn-benchmark ile yeniden"); continue
        break
    print(f"[{name}] BAŞARISIZ (çıkış {rc}); günlük: {LOGS}/{name}.log")
    return rc

def run_lane(lane, gpu, workers):
    cache = f"{CACHE_ROOT}/cache_{lane}"
    for job in LANES[lane]:
        t = time.time(); rc = train_job(job, gpu, cache, workers)
        print(f"[şerit {lane}] {job[0]} bitti: çıkış {rc}, {(time.time() - t) / 60:.0f} dk", flush=True)


In [ ]:
# 4) KURU KOŞU (MODE == "dry"): test'e dokunmadan hat + paralel verim + kilit testleri
DRY = {}
if MODE == "dry":
    D = "/kaggle/working/dry"; os.makedirs(D, exist_ok=True)
    t0 = time.time()
    # D1: iki GPU'da aynı anda 2 epoch pilot (Malatya katlaması, pilotla aynı yol) → paralel verim ölçümü
    def dry_par(i):
        DRY[f"d{i}"] = run_job(f"dry_d{i}", ["--scenario", "S1", "--holdout", "Malatya", "--pilot", "--pilot-tiles", "64",
                                            "--epochs", "2", "--seed", str(42 + i), "--out", f"{D}/d{i}"], i, f"{CACHE_ROOT}/cache_dry{i}", 2)
    th = [threading.Thread(target=dry_par, args=(i,)) for i in (0, 1)]
    [t.start() for t in th]; [t.join() for t in th]
    DRY["par_wall_min"] = (time.time() - t0) / 60
    # D2: tam yol (holdout yok → eğitim içi sanity) S2, tek GPU
    DRY["s2"] = run_job("dry_s2", ["--scenario", "S2", "--pilot", "--pilot-tiles", "16", "--epochs", "1", "--out", f"{D}/s2"], 0, f"{CACHE_ROOT}/cache_dry0", 4)
    # D3: yalnız-değerlendirme yolu, KONTROL rolü (Gaziantep, test değil), 4 karo
    DRY["ctrl"] = run_job("dry_ctrl", ["--scenario", "S1", "--eval-only", "--eval-role", "control", "--eval-max-tiles", "4",
                                       "--ckpt", f"{D}/d0/model_final.pt", "--out", f"{D}/ctrl"], 0, f"{CACHE_ROOT}/cache_dry0", 2)
    # D4: kilitler (başarısız olmaları BEKLENİR): test --allow-test'siz, Osmaniye --trigger-fold'suz
    def lock(args):
        r = subprocess.run([sys.executable, "-m", "src.train_changeos", *args, "--out", f"{D}/lock"], cwd=REPO, capture_output=True, text=True,
                           env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"))
        return r.returncode, (r.stdout + r.stderr)
    rc1, o1 = lock(["--scenario", "S1", "--eval-only", "--eval-role", "test", "--ckpt", "yok.pt"])
    rc2, o2 = lock(["--scenario", "S1", "--holdout", "Osmaniye", "--epochs", "1"])
    DRY["lock_test_ok"] = rc1 != 0 and "kilitli" in o1
    DRY["lock_osm_ok"] = rc2 != 0 and "trigger-fold" in o2
    print("test kilidi:", "ÇALIŞIYOR" if DRY["lock_test_ok"] else "SORUN!\n" + o1[-400:])
    print("Osmaniye kilidi:", "ÇALIŞIYOR" if DRY["lock_osm_ok"] else "SORUN!\n" + o2[-400:])
else:
    print("MODE != dry: kuru koşu atlandı")


In [ ]:
# 5) KURU KOŞU ÖZETİ
if MODE == "dry":
    REF = 11.5   # pilot Aşama 0 tek-GPU sabit-durum kırpım/sn
    print("=== KURU KOŞU ÖZETİ ===")
    print("çıkış kodları:", {k: v if not isinstance(v, tuple) else v[0] for k, v in DRY.items() if k in ("d0", "d1", "s2", "ctrl")})
    ok_all = True
    speeds = []
    for i in (0, 1):
        lg = f"/kaggle/working/dry/d{i}/log.jsonl"
        if os.path.exists(lg):
            rec = [json.loads(l) for l in open(lg).read().strip().splitlines()][-1]
            speeds.append(rec["crops_per_s"])
            print(f"GPU{i}: epoch {rec['epoch']} {rec['sec']} sn | {rec['crops_per_s']} kırpım/sn | veri bekleme {rec['data_wait_s']}/{rec['sec']} sn | tepe bellek {rec['peak_mem_GB']} GB")
        else:
            ok_all = False; print(f"GPU{i}: log yok → SORUN")
    print(f"paralel bölüm duvar süresi: {DRY['par_wall_min']:.1f} dk  (kota göstergesiyle karşılaştır, aşağıya bak)")
    for k in ("d0", "d1", "s2", "ctrl"):
        if DRY[k][0] != 0: ok_all = False
    if speeds:
        verdict = "paralel MAKUL (PARALLEL=True)" if min(speeds) >= 0.75 * REF else "paralel VERİMSİZ → PARALLEL=False, şeritleri ayrı sürümlerde koş"
        print(f"referans tek-GPU {REF} kırpım/sn; paralelde en yavaş {min(speeds)} → {verdict}")
        print("NOT: epoch 2 bellek/önbellek ısınması sonrası; tam eğitimde sabit-durum epoch daha net ölçülür.")
    ok_all = ok_all and DRY.get("lock_test_ok") and DRY.get("lock_osm_ok")
    print("HAT:", "TAMAM → MODE='run' yapılabilir" if ok_all else "SORUN VAR → run'a geçme, yukarıdaki çıktıyı bana yapıştır")
    print("""
KOTA KONTROLÜ (elle): Kaggle sağ üst → profil → 'Your Kaggle Quota' → GPU. Kuru koşudan ÖNCEKİ değer ile SONRAKİ değerin farkı,
yukarıdaki 'paralel bölüm duvar süresi'ne (~1x) yakınsa kota OTURUM süresine göre işliyor (paralel koşu kotayı yarıya indirir);
~2x ise GPU başınadır (paralel kazanç yalnız duvar süresinde). Farkı bana yaz.""")


In [ ]:
# 6) TAM EĞİTİM (MODE == "run")
if MODE == "run":
    t0 = time.time()
    if PARALLEL and len(RUN_LANES) > 1:
        th = [threading.Thread(target=run_lane, args=(l, i, 2)) for i, l in enumerate(RUN_LANES)]
        [t.start() for t in th]; [t.join() for t in th]
    else:
        for l in RUN_LANES:
            run_lane(l, 0, 4)
    print(f"TOPLAM duvar süresi: {(time.time() - t0) / 3600:.2f} sa")
else:
    print("MODE != run: tam eğitim atlandı")


In [ ]:
# 7) Özet ve temizlik (yalnız run): eğitim-içi SANITY ve tetikleyici (Osmaniye katlaması) — TEST YOK
if MODE == "run":
    sh(f"python -m src.summarize_runs {RUNS}", check=False, tail=6000)
    p = f"{RUNS}/trigger_osmaniye_seed42/metrics.json"
    if os.path.exists(p):
        print("TETİKLEYİCİ (Osmaniye katlaması, ön-kayıtlı asgari-skor kurallı):", json.load(open(p)).get("trigger"))
    for d in glob.glob(f"{RUNS}/*/"):
        if os.path.exists(d + "model_final.pt") and os.path.exists(d + "last.pt"):
            os.remove(d + "last.pt")        # Output boyutu için (koşu tamam; resume gerekmez)
    sh("du -sh /kaggle/working/runs /kaggle/working/logs | cat")
    print("Bitti. TEST DEĞERLENDİRMESİ bu not defterinde YOK: ayrı turda (ön-kayıt §4).")
